Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public
License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later
version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied
warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see
<https://www.gnu.org/licenses/>.

# F3: Memories as Golay Codewords

*A hunch* ([F1-F9.md § F3](../docs/hypotheses/experiments/F1-F9.md#f3-memories-as-codewords-of-an-error-correcting-code)):
memories stored as codewords of a highly symmetric code are recalled more robustly from corrupted cues. Three stores of
64 words of 24 bits - 64 codewords of the extended Golay code, a matched store of random words at least 8 apart, and an
unstructured store of random words - recalled by a modern Hopfield network ([Ramsauer et al.
2020](https://arxiv.org/abs/2008.02217)), a softmax over the stored words at inverse temperature β. Everything the plan
left open was fixed in the journal of 2026-11-05, § F3 Run: Memories as Golay Codewords, before the test seeds were
run. This explores; nothing the benchmark depends on lives here.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")  # one BLAS thread per process: two processes in all
import math, itertools, time, platform
import multiprocessing as mp
import numpy as np
import matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt
print("python", platform.python_version(), "numpy", np.__version__, "matplotlib", matplotlib.__version__)

python 3.13.16 numpy 2.5.4 matplotlib 3.11.2


## The Model, the Stores and the Readout

Words are coded ±1. One step is ξ ← Xᵀ softmax(β X ξ); a cue settles when no coordinate moves by more than 10⁻¹², at
most 100 steps, and an unsettled cue is a failure. The readout rounds each coordinate of the final state (one within
10⁻⁹ of zero is undecided and counts one half towards either value) and takes the nearest stored word; a tie is a
failure.

In [2]:
import numpy as np, itertools, math, time
N = 24
G_POLY = [0, 2, 4, 5, 6, 10, 11]  # exponents of g(x) = x^11+x^10+x^6+x^5+x^4+x^2+1

def golay_code():
    g = 0
    for e in G_POLY: g |= 1 << e
    words = []
    for m in range(1 << 12):
        c = 0
        for i in range(12):
            if (m >> i) & 1: c ^= g << i
        bits = [(c >> i) & 1 for i in range(23)]
        bits.append(sum(bits) % 2)
        words.append(bits)
    return np.array(words, dtype=np.int8)

def pm(b):  # 0/1 -> -1/+1
    return 2.0 * b.astype(np.float64) - 1.0

def settle(P, C, beta, maxit=100, tol=1e-12, chunk=65536):
    """Iterate xi <- P^T softmax(beta P xi) per cue until no coordinate moves by more than tol, at most maxit steps."""
    out = np.empty_like(C); unsettled = np.zeros(len(C), dtype=bool)
    for s in range(0, len(C), chunk):
        xi = C[s:s+chunk].copy(); act = np.arange(len(xi))
        for _ in range(maxit):
            x = xi[act]
            L = beta * (x @ P.T)
            L -= L.max(axis=1, keepdims=True)
            W = np.exp(L); W /= W.sum(axis=1, keepdims=True)
            new = W @ P
            moving = np.abs(new - x).max(axis=1) > tol
            xi[act] = new
            act = act[moving]
            if len(act) == 0: break
        unsettled[s + act] = True
        out[s:s+chunk] = xi
    return out, unsettled

def readout(P, X, own, eps=1e-9):
    r = np.where(X > eps, 1, np.where(X < -eps, -1, 0)).astype(np.int64)
    Pi = P.astype(np.int64)
    twice_d = N - r @ Pi.T                     # 2 x Hamming distance, undecided counting 1/2
    best = twice_d.min(axis=1, keepdims=True)
    nbest = (twice_d == best).sum(axis=1)
    arg = twice_d.argmin(axis=1)
    return (nbest == 1) & (arg == own)

def recall(P, C, own, beta, sign=1.0):
    """Recalled per cue; a cue not settled within the step cap counts as a failure."""
    X, unsettled = settle(P, C, sign * beta)
    return readout(P, X, own) & ~unsettled, int(unsettled.sum())

FLIPSETS = {f: np.array(list(itertools.combinations(range(N), f)), dtype=np.int64).reshape(math.comb(N, f), f) for f in range(5)}

def flip_cues(P, sets):  # sets: (n_sets,f) shared, or (n_words, n_sets, f)
    n = len(P)
    if sets.ndim == 2:
        sets = np.broadcast_to(sets, (n,) + sets.shape)
    ns, f = sets.shape[1], sets.shape[2]
    C = np.repeat(P[:, None, :], ns, axis=1).copy()
    if f:
        ii = np.arange(n)[:, None, None]; jj = np.arange(ns)[None, :, None]
        C[ii, jj, sets] *= -1
    return C.reshape(-1, N), np.repeat(np.arange(n), ns)

def erase_cues(P, given):  # given: (n_sets, k) positions kept
    n = len(P); ns = len(given)
    mask = np.zeros((ns, N)); mask[np.arange(ns)[:, None], given] = 1
    C = (P[:, None, :] * mask[None, :, :]).reshape(-1, N)
    return C, np.repeat(np.arange(n), ns)

def draw_golay(code, seed):
    rng = np.random.default_rng([seed, 1])
    return code[rng.choice(len(code), 64, replace=False)]

def draw_matched(seed, dmin=8):
    rng = np.random.default_rng([seed, 2]); kept = []; draws = 0
    while len(kept) < 64:
        w = rng.integers(0, 2, N, dtype=np.int8); draws += 1
        if all(int((w != k).sum()) >= dmin for k in kept): kept.append(w)
    return np.array(kept), draws

def draw_unstructured(seed):
    return np.random.default_rng([seed, 3]).integers(0, 2, (64, N), dtype=np.int8)

def draw_pair(code, seed):
    rng = np.random.default_rng([seed, 4])
    c = code[rng.integers(len(code))]
    a = code[rng.choice(np.flatnonzero(code.sum(1) == 8))]
    return np.stack([c, c ^ a])  # two codewords at distance 8

def pair_dists(B):
    d = (B[:, None, :] != B[None, :, :]).sum(-1)
    return d[np.triu_indices(len(B), 1)]


## The Code

The 4096 multiples of g(x) = x¹¹ + x¹⁰ + x⁶ + x⁵ + x⁴ + x² + 1 of degree below 23, each extended by a parity bit. The
weight distribution must be 1, 759, 2576, 759, 1 at weights 0, 8, 12, 16, 24; for a linear code, the smallest nonzero
weight is the minimum distance.

In [3]:
CODE = golay_code()
wd = np.bincount(CODE.sum(1), minlength=N + 1)
print("distinct codewords:", len({r.tobytes() for r in CODE}))
print("weight distribution:", {w: int(c) for w, c in enumerate(wd) if c})
assert len({r.tobytes() for r in CODE}) == 4096
assert {w: int(c) for w, c in enumerate(wd) if c} == {0: 1, 8: 759, 12: 2576, 16: 759, 24: 1}
# closed under addition (linearity), checked on 2000 random pairs
rng = np.random.default_rng(0)
cs = {r.tobytes() for r in CODE}
assert all((CODE[i] ^ CODE[j]).tobytes() in cs for i, j in rng.integers(0, 4096, (2000, 2)))
print("checks passed")

distinct codewords: 4096
weight distribution: {0: 1, 8: 759, 12: 2576, 16: 759, 24: 1}
checks passed


## The Harness Check, and Its Red-First Breaks

Two codewords at distance 8, every cue with f = 3 flipped. Below β = 1/8 the mean of the two is a stable fixed point
(journal, the correction under § F3 Run), so a low β must fail. Shown on the probe seeds over the β grid, then the
breaks: β = 0.084, the largest grid value at which every cue settles on the mean within the 100 steps, and the logits'
sign reversed. Then β = 0.119, which shows why an unsettled cue counts as a failure: just below 1/8 the drift to the
mean is too slow for the cap, and the readout alone would pass.

In [4]:
BETAS = 0.05 * 2.0 ** (np.arange(49) / 4)
PROBE = [1001, 1002, 1003]

def all_f3(B, beta, sign=1.0):
    P = pm(B); C, own = flip_cues(P, FLIPSETS[3]); ok, uns = recall(P, C, own, beta, sign)
    return bool(ok.all()), float(ok.mean()), uns

for s in PROBE:
    pair = draw_pair(CODE, s)
    assert (pair[0] != pair[1]).sum() == 8
    row = [(round(b, 3), all_f3(pair, b)[1]) for b in BETAS[:10]]
    print(s, "recall of the pair at f = 3 by beta:", row)
pair = draw_pair(CODE, 1001)
lo = float(BETAS[3])
br1 = all_f3(pair, lo)
br2 = all_f3(pair, 1.0, sign=-1.0)
print(f"break 1, beta = {lo:.4f}: all recalled {br1[0]}, recall {br1[1]:.3f}, unsettled {br1[2]}")
print(f"break 2, sign reversed at beta = 1: all recalled {br2[0]}, recall {br2[1]:.3f}, unsettled {br2[2]}")
assert not br1[0] and br1[2] == 0, "break 1 did not fail by settling on the mean"
assert not br2[0], "break 2 did not fail the check"
assert all_f3(pair, 1.0)[0], "the check fails at beta = 1"
print("both breaks fail the check; the unbroken check passes at beta = 1")
cap = float(BETAS[5]); P2 = pm(pair); C2, own2 = flip_cues(P2, FLIPSETS[3]); X2, u2 = settle(P2, C2, cap)
print(f"the cap rule, beta = {cap:.4f}: unsettled {int(u2.sum())} of {len(C2)}; readout alone recalls "
      f"{readout(P2, X2, own2).mean():.3f}, with unsettled cues as failures {all_f3(pair, cap)[1]:.3f}")

1001 recall of the pair at f = 3 by beta: [(np.float64(0.05), 0.0), (np.float64(0.059), 0.0), (np.float64(0.071), 0.0), (np.float64(0.084), 0.0), (np.float64(0.1), 0.0), (np.float64(0.119), 0.0), (np.float64(0.141), 0.9723320158102767), (np.float64(0.168), 1.0), (np.float64(0.2), 1.0), (np.float64(0.238), 1.0)]


1002 recall of the pair at f = 3 by beta: [(np.float64(0.05), 0.0), (np.float64(0.059), 0.0), (np.float64(0.071), 0.0), (np.float64(0.084), 0.0), (np.float64(0.1), 0.0), (np.float64(0.119), 0.0), (np.float64(0.141), 0.9723320158102767), (np.float64(0.168), 1.0), (np.float64(0.2), 1.0), (np.float64(0.238), 1.0)]


1003 recall of the pair at f = 3 by beta: [(np.float64(0.05), 0.0), (np.float64(0.059), 0.0), (np.float64(0.071), 0.0), (np.float64(0.084), 0.0), (np.float64(0.1), 0.0), (np.float64(0.119), 0.0), (np.float64(0.141), 0.9723320158102767), (np.float64(0.168), 1.0), (np.float64(0.2), 1.0), (np.float64(0.238), 1.0)]
break 1, beta = 0.0841: all recalled False, recall 0.000, unsettled 0
break 2, sign reversed at beta = 1: all recalled False, recall 0.000, unsettled 4048
both breaks fail the check; the unbroken check passes at beta = 1


the cap rule, beta = 0.1189: unsettled 4048 of 4048; readout alone recalls 1.000, with unsettled cues as failures 0.000


## Choosing β on the Probe Seeds

The smallest β on the grid at which, on each probe seed, the pair, the Golay store and the matched store all recall
every f = 3 cue.

In [5]:
def controls_hold(beta):
    for s in PROBE:
        stores = [draw_pair(CODE, s), draw_golay(CODE, s), draw_matched(s)[0]]
        for B in stores:
            if not all_f3(B, beta)[0]:
                return False
    return True

t0 = time.time()
BETA = None
for b in BETAS:
    if controls_hold(b):
        BETA = float(b); break
print(f"chosen beta = {BETA:.4f} (grid index {int(np.argmin(abs(BETAS - BETA)))}), in {time.time() - t0:.0f} s")
prev = float(BETAS[BETAS < BETA].max())
for s in PROBE:
    print(s, "at the grid value below,", round(prev, 4), ": pair, Golay, matched all-recalled =",
          [all_f3(B, prev)[0] for B in (draw_pair(CODE, s), draw_golay(CODE, s), draw_matched(s)[0])])

chosen beta = 0.4000 (grid index 12), in 53 s


1001 at the grid value below, 0.3364 : pair, Golay, matched all-recalled = [True, False, True]


1002 at the grid value below, 0.3364 : pair, Golay, matched all-recalled = [True, False, True]


1003 at the grid value below, 0.3364 : pair, Golay, matched all-recalled = [True, False, True]


### Probe Seed 1001 at Larger β

On probe seed 1001 only (not a test seed): the Golay store's recall at f = 4 at the chosen β and at two larger ones, for
the journal's remark on how recall moves with β. Not a test.

In [6]:
Pg = pm(draw_golay(CODE, 1001)); Cg, owng = flip_cues(Pg, FLIPSETS[4])
for b in (BETA, 1.0, 4.0):
    ok, uns = recall(Pg, Cg, owng, b)
    print(f"probe seed 1001, Golay store, f = 4, beta = {b:g}: recall {ok.mean():.4f}, unsettled {uns}")

probe seed 1001, Golay store, f = 4, beta = 0.4: recall 0.9601, unsettled 844


probe seed 1001, Golay store, f = 4, beta = 1: recall 0.9613, unsettled 0


probe seed 1001, Golay store, f = 4, beta = 4: recall 0.9422, unsettled 0


## The Test Seeds, 1 to 20

Per seed and store: recall at f = 0 to 6, the unsettled cues, recall over the 100 sets of given positions at each k =
10 to 24, and the stored words' pairwise distances. Two processes.

In [7]:
TEST = list(range(1, 21))
KS = list(range(10, 25))

def run_seed(seed):
    rng5 = np.random.default_rng([seed, 5])
    sampled = {f: rng5.random((64, 1000, N)).argsort(axis=-1)[..., :f] for f in (5, 6)}
    rng6 = np.random.default_rng([seed, 6])
    given = {k: rng6.random((100, N)).argsort(axis=1)[:, :k] for k in KS}
    m, draws = draw_matched(seed)
    stores = {"golay": draw_golay(CODE, seed), "matched": m, "unstructured": draw_unstructured(seed)}
    out = {"matched_draws": draws}
    for name, B in stores.items():
        P = pm(B); r = {"recall_f": [], "unsettled_f": [], "recall_k": [], "per_set_k12": None}
        for f in range(7):
            C, own = flip_cues(P, FLIPSETS[f] if f < 5 else sampled[f])
            ok, uns = recall(P, C, own, BETA)
            r["recall_f"].append(float(ok.mean())); r["unsettled_f"].append(uns)
        for k in KS:
            C, own = erase_cues(P, given[k])
            ok, uns = recall(P, C, own, BETA)
            per_set = ok.reshape(64, 100).mean(axis=0)
            r["recall_k"].append(float(ok.mean()))
            if k == 12: r["per_set_k12"] = per_set
        r["dists"] = np.bincount(pair_dists(B), minlength=N + 1)
        out[name] = r
    return out

t0 = time.time()
with mp.get_context("fork").Pool(2) as pool:
    RES = pool.map(run_seed, TEST)
print(f"{len(RES)} seeds in {time.time() - t0:.0f} s")
STORES = ["golay", "matched", "unstructured"]
RF = {s: np.array([r[s]["recall_f"] for r in RES]) for s in STORES}
UF = {s: np.array([r[s]["unsettled_f"] for r in RES]) for s in STORES}
RK = {s: np.array([r[s]["recall_k"] for r in RES]) for s in STORES}
print("matched-store draws per seed:", [r["matched_draws"] for r in RES])
print("unsettled cues, summed over seeds, by f:", {s: UF[s].sum(0).tolist() for s in STORES})

20 seeds in 900 s
matched-store draws per seed: [200, 314, 323, 238, 254, 251, 217, 219, 214, 219, 226, 224, 233, 225, 275, 216, 305, 273, 170, 206]
unsettled cues, summed over seeds, by f: {'golay': [0, 0, 0, 0, 29772, 26325, 360856], 'matched': [0, 0, 0, 0, 121, 21677, 293600], 'unstructured': [0, 0, 32, 566, 5693, 9459, 115806]}


## The Two Refuting Clauses

Paired by seed, percentile bootstrap over the 20 seeds, 10 000 resamples. Refuted when the one-sided upper 97.5% bound
is at or below zero; passed when the one-sided lower 95% bound is above zero; otherwise inconclusive.

In [8]:
def boot_means(d, n=10_000, seed=20261105):
    idx = np.random.default_rng(seed).integers(0, len(d), (n, len(d)))
    return d[idx].mean(axis=1)

def decide(d):
    bm = boot_means(d)
    up, lo = np.quantile(bm, 0.975), np.quantile(bm, 0.05)
    verdict = "refuted" if up <= 0 else ("passed" if lo > 0 else "inconclusive")
    return float(d.mean()), float(lo), float(up), verdict

for f in (3, 4):
    print(f"f = {f}: mean recall " + ", ".join(f"{s} {RF[s][:, f].mean():.4f} [{RF[s][:, f].min():.4f}, {RF[s][:, f].max():.4f}]" for s in STORES))
a = decide(RF["golay"][:, 3] - RF["unstructured"][:, 3])
b = decide(RF["golay"][:, 4] - RF["matched"][:, 4])
print("(a) f = 3, Golay - unstructured: mean %+.4f, lower 95%% %+.4f, upper 97.5%% %+.4f -> %s" % a)
print("(b) f = 4, Golay - matched:      mean %+.4f, lower 95%% %+.4f, upper 97.5%% %+.4f -> %s" % b)
print("assertion, every f <= 3 cue recalled in both stores of minimum distance 8 on every test seed:",
      bool((RF["golay"][:, :4] == 1).all() and (RF["matched"][:, :4] == 1).all()))

f = 3: mean recall golay 1.0000 [1.0000, 1.0000], matched 1.0000 [1.0000, 1.0000], unstructured 0.9954 [0.9908, 0.9972]
f = 4: mean recall golay 0.9604 [0.9489, 0.9644], matched 0.9895 [0.9876, 0.9909], unstructured 0.9725 [0.9653, 0.9781]
(a) f = 3, Golay - unstructured: mean +0.0046, lower 95% +0.0041, upper 97.5% +0.0054 -> passed
(b) f = 4, Golay - matched:      mean -0.0290, lower 95% -0.0305, upper 97.5% -0.0276 -> refuted
assertion, every f <= 3 cue recalled in both stores of minimum distance 8 on every test seed: True


## Why f = 4 Comes Out as It Does - Post Hoc

Not pre-registered: added after the verdict, to explain it. At f = 4 a cue can lie 4 from its own word and 4 from a
stored word 8 away, tied by distance. Counted here: the stored pairs at distance exactly 8 per store; and, on test
seed 1, the share of f = 4 cues tied by distance, with the failure rate among tied and among untied cues, per store.
The softmax still weighs the other stored words, so a tie by distance need not be a failure.

In [9]:
def ties_at_f4(B):
    P = pm(B); C, own = flip_cues(P, FLIPSETS[4])
    d = (N - C.astype(np.int64) @ P.astype(np.int64).T) // 2
    dn = d.copy(); dn[np.arange(len(d)), own] = N + 1
    tied = dn.min(axis=1) == 4
    fail = ~recall(P, C, own, BETA)[0]
    return tied.mean(), fail[tied].mean(), fail[~tied].mean(), fail.mean()

for s in ("golay", "matched"):
    n8 = np.array([r[s]["dists"][8] for r in RES])
    print(f"{s}: pairs at distance 8 per store, mean {n8.mean():.1f} [{n8.min()}, {n8.max()}]")
for s, B in (("golay", draw_golay(CODE, 1)), ("matched", draw_matched(1)[0]), ("unstructured", draw_unstructured(1))):
    t, ft, fu, fa = ties_at_f4(B)
    print(f"seed 1, f = 4, {s}: tied by distance {t:.4f}; failure among tied {ft:.4f}, among untied {fu:.4f}; all {fa:.4f}")

golay: pairs at distance 8 per store, mean 377.2 [351, 414]
matched: pairs at distance 8 per store, mean 102.7 [89, 121]


seed 1, f = 4, golay: tied by distance 0.0727; failure among tied 0.5167, among untied 0.0000; all 0.0376


seed 1, f = 4, matched: tied by distance 0.0225; failure among tied 0.5009, among untied 0.0000; all 0.0113


seed 1, f = 4, unstructured: tied by distance 0.0412; failure among tied 0.5114, among untied 0.0104; all 0.0311


## Reported, Not Tested

The stores' distance histograms pooled over the seeds; recall over every f and k; and the variance of recall across
the 100 sets of given positions at k = 12.

In [10]:
for s in STORES:
    h = sum(r[s]["dists"] for r in RES)
    print(s, "pairwise distances:", {d: int(c) for d, c in enumerate(h) if c})
print("recall by f (mean over seeds):")
for s in STORES: print(f"  {s:12s}", np.round(RF[s].mean(0), 4).tolist())
print("recall by k = 10..24 (mean over seeds):")
for s in STORES: print(f"  {s:12s}", np.round(RK[s].mean(0), 4).tolist())
print("variance of recall across the 100 sets at k = 12, mean over seeds [min, max]:")
for s in STORES:
    v = np.array([r[s]["per_set_k12"].var() for r in RES])
    print(f"  {s:12s} {v.mean():.5f} [{v.min():.5f}, {v.max():.5f}]")

golay pairwise distances: {8: 7545, 12: 25391, 16: 7380, 24: 4}
matched pairwise distances: {8: 2053, 9: 3324, 10: 5173, 11: 6103, 12: 6664, 13: 6123, 14: 4738, 15: 3079, 16: 1744, 17: 842, 18: 333, 19: 109, 20: 31, 21: 3, 22: 1}
unstructured pairwise distances: {3: 5, 4: 25, 5: 104, 6: 278, 7: 806, 8: 1764, 9: 3144, 10: 4715, 11: 5925, 12: 6463, 13: 6071, 14: 4683, 15: 3242, 16: 1762, 17: 840, 18: 352, 19: 103, 20: 31, 21: 7}
recall by f (mean over seeds):
  golay        [1.0, 1.0, 1.0, 1.0, 0.9604, 0.8293, 0.3557]
  matched      [1.0, 1.0, 1.0, 1.0, 0.9895, 0.9007, 0.3393]
  unstructured [1.0, 1.0, 0.9994, 0.9954, 0.9725, 0.8597, 0.3897]
recall by k = 10..24 (mean over seeds):
  golay        [0.6267, 0.9468, 0.993, 0.9984, 0.9995, 0.9999, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  matched      [0.6711, 0.989, 0.9981, 0.9995, 0.9999, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  unstructured [0.683, 0.9498, 0.989, 0.9952, 0.9978, 0.9989, 0.9993, 0.9998, 0.9999, 0.9999, 1.0, 

## Figure

Mean recall over the 20 seeds, with the range over seeds as a band. Written to `docs/figures/f3_golay_recall.svg`.

In [11]:
plt.rcParams["svg.hashsalt"] = "f3-golay"
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
style = {"golay": ("#1f6feb", "Golay"), "matched": ("#d29922", "matched (min. distance 8)"), "unstructured": ("#8b949e", "unstructured")}
for s in STORES:
    c, lab = style[s]
    ax[0].plot(range(7), RF[s].mean(0), "o-", color=c, label=lab)
    ax[0].fill_between(range(7), RF[s].min(0), RF[s].max(0), color=c, alpha=0.2, lw=0)
    ax[1].plot(KS, RK[s].mean(0), "o-", color=c, label=lab, ms=3)
    ax[1].fill_between(KS, RK[s].min(0), RK[s].max(0), color=c, alpha=0.2, lw=0)
ax[0].set_xlabel("features flipped, f"); ax[0].set_ylabel("recall"); ax[0].set_ylim(0, 1.02)
ax[1].set_xlabel("features given, k (rest erased)"); ax[1].set_ylim(0, 1.02)
ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title(f"flipped cues, beta = {BETA:.3f}", fontsize=9); ax[1].set_title("erased cues", fontsize=9)
fig.tight_layout()
fig.savefig("../docs/figures/f3_golay_recall.svg", metadata={"Date": None})
plt.close(fig)
print("written")

written
